# Capstone Framing — Structured Content Archetype Clustering

**Lane:** Structured Content Archetype Clustering
**Main question:** What performance archetypes exist across the content inventory?

This notebook establishes the lane, the decision the analysis is meant to support, real evidence from the starter dataset, and honest claim boundaries. It does **not** include modeling, clustering, or validation — those belong to later weeks (W05/W06).

## 1. My Lane and Why

My lane is Structured Content Archetype Clustering. I want to identify recurring types of content pages based on their search demand, visibility, freshness, content size, and engagement patterns. Instead of reviewing every page independently, clustering can group content items that show similar observed patterns and make the content inventory easier to understand. The result can support decisions about which pages should be protected, improved, rewritten, or monitored. I chose this lane because it directly addresses content portfolio prioritization while allowing the analysis to remain descriptive and evidence-based.

This is **structured / metric-based content archetype clustering**, not semantic clustering — the warehouse does not contain article text, so grouping is based on content metadata and search/engagement signals rather than content meaning.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(len(df))
print(df.columns.tolist())
print(df["client_id"].nunique())

## 2. The Question: Decision, Action, Cost

The main decision is which type of content should be reviewed first and what kind of review may be appropriate. A content or SEO team would use the output to prioritize limited review time across the content inventory. The possible actions are Protect, Improve, Rewrite, Monitor, or Review, depending on the observed archetype. A wrong recommendation has an opportunity cost: rewriting strong content can waste time and introduce unnecessary changes, while failing to review stale or weak content can cause useful opportunities to be missed. Therefore, the model is intended to support human prioritization rather than automatically decide what change will succeed.

In [ ]:
required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
]

missing_columns = [c for c in required_columns if c not in df.columns]

print(required_columns)
print(missing_columns)
print(len(missing_columns) == 0)

## 3. Quick Look at the Data

The starter dataset contains 30,000 content items across 32 clients. This is enough content and client variation to investigate recurring patterns instead of looking at isolated examples. There are also 1,205 rows where `avg_position` equals zero. The dataset guidance defines this as no position data, not a genuine ranking, which shows that missing-data handling will be an important part of the later modeling pipeline. These observations make the lane worth developing further over the next several weeks.

In [ ]:
n_rows = len(df)
n_clients = df["client_id"].nunique()
n_zero_position = (df["avg_position"] == 0).sum()

print(n_rows)
print(n_clients)
print(n_zero_position)

## 4. Careful Words

This work can report observed and measured differences between content groups. It can provide directional evidence about which content characteristics and performance signals are associated with different archetypes. The output can be used as decision-support for content review, prioritization, and monitoring.

This work cannot prove that rewriting or updating a page will increase traffic, prove what causes search-engine rankings, or predict Google's ranking algorithm. The clusters are a lens for organizing observed patterns in the available data, not true labels and not causal conclusions.

In [ ]:
safe_words = [
    "observed",
    "measured",
    "directional",
    "associated with",
    "suggests",
    "decision-support",
    "in this dataset",
    "in this snapshot",
]

avoid_words = [
    "proves",
    "causes",
    "guarantees",
    "will increase",
    "definitely improves",
    "predicts Google",
    "ranking factor",
    "guaranteed recovery",
]

# What this work CAN claim - should use safe words only
claims_made = (
    "This work can report observed and measured differences between content groups. "
    "It can provide directional evidence about which content characteristics and performance "
    "signals are associated with different archetypes. The output can be used as decision-support "
    "for content review, prioritization, and monitoring."
)

# What this work explicitly CANNOT claim - avoid words are expected here, since this is
# the boundary statement that names and disclaims them
boundaries_stated = (
    "This work cannot prove that rewriting or updating a page will increase traffic, "
    "prove what causes search-engine rankings, or predict Google's ranking algorithm."
)

safe_used_in_claims = [w for w in safe_words if w.lower() in claims_made.lower()]
avoid_found_in_claims = [w for w in avoid_words if w.lower() in claims_made.lower()]
avoid_named_in_boundaries = [w for w in avoid_words if w.lower() in boundaries_stated.lower()]

print("Safe words used in the claims statement:", safe_used_in_claims)
print("Avoid-words found in the claims statement (should be empty):", avoid_found_in_claims)
print("Avoid-words explicitly named in the boundaries statement (expected, not a violation):", avoid_named_in_boundaries)
print("\nClaims statement is public-safe:", len(avoid_found_in_claims) == 0)

## Self-Check

In [ ]:
self_checks = []

self_checks.append(("Every section contains a Markdown explanation", True))  # Sections 1-4 each open with markdown
self_checks.append(("Every section has supporting code/checks", True))       # each section has a code cell

self_checks.append(("Starter CSV loads successfully", len(df) > 0))
self_checks.append(("Three real data numbers calculated by code (not hard-coded)",
                     n_rows == len(df) and n_clients == df["client_id"].nunique() and n_zero_position == (df["avg_position"] == 0).sum()))

name_exposing_columns = [c for c in df.columns if any(
    term in c.lower() for term in ["client_name", "company", "domain", "brand"]
)]
self_checks.append(("No client-name-exposing columns present", len(name_exposing_columns) == 0))
self_checks.append(("No URLs exposed", not any("url" in c.lower() for c in df.columns)))
self_checks.append(("No private/raw queries shown", not any("query" in c.lower() for c in df.columns)))

self_checks.append(("Claims use observed/measured/directional/decision-support language", len(avoid_found_in_claims) == 0))

print("=== Self-Check ===\n")
all_passed = True
for label, passed in self_checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"{status} — {label}")

print()
if all_passed:
    print("All automated checks PASS.")
else:
    print("One or more checks FAILED — review above before submission.")

print("\nManual checklist (confirm yourself):")
print("  [ ] Notebook runs top to bottom without errors on a fresh kernel")
print("  [ ] Notebook is saved under work/notebooks/")
print("  [ ] Notebook is committed to the repository")
print("  [ ] Repository URL is submitted on the assignment card")